# 03 — Preparación de Features

**v2 no tiene feature engineering adicional** (sin rolling M1 ni cross-sensor M2 de v3).
Este notebook solo selecciona las columnas numéricas relevantes y genera X e y para los modelos.

**Entrada:** `data/interim/02_datos_inyectados.parquet`  
**Salida:** `data/interim/03_datos_features.parquet`

In [ ]:
# ─── Intel Extension for Scikit-learn ────────────────────────────────────────
# IMPORTANTE: debe cargarse ANTES que cualquier import de sklearn.
# Instalar con: pip install scikit-learn-intelex
# En CPU Intel puede acelerar Random Forest e IterativeImputer 10-100x.
try:
    from sklearnex import patch_sklearn
    patch_sklearn()
    _intel_activo = True
except ImportError:
    _intel_activo = False

# ─── Librerías estándar ───────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import os, glob, joblib, pickle, warnings
import matplotlib.ticker as mticker

# ─── Scikit-learn (ya parcheado si Intel Extension está disponible) ───────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.experimental import enable_iterative_imputer  # requerido antes de IterativeImputer
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                              classification_report, confusion_matrix,
                              average_precision_score)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

# ─── Configuración del proyecto ───────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import *

if _intel_activo:
    print("Intel Extension for Scikit-learn ACTIVA — algoritmos acelerados con oneAPI")
else:
    print("Tip: instala scikit-learn-intelex para acelerar en CPU Intel")
    print("     pip install scikit-learn-intelex")


## 0. Cargar datos del paso anterior

In [ ]:
df_trabajo = pd.read_parquet(PARQUET_02)
print(f"Dataset cargado: {df_trabajo.shape}")


## 1. Selección de features
Excluimos Fecha y columnas de etiquetas. El resto son las features de entrada para los modelos.

**v2:** sin features de rolling ni cross-sensor. Solo las columnas originales del dataset + temporales (Hora, DiaSemana, Mes).

In [ ]:
if 'df_trabajo' in locals():
    excluir = ['Fecha', 'etiqueta_deteccion', 'etiqueta_tipo_anomalia']
    columnas_potenciales_features = [c for c in df_trabajo.columns if c not in excluir]

    X = df_trabajo[columnas_potenciales_features].select_dtypes(include='number').copy()
    print(f'Características (X): {X.shape[1]} columnas numéricas')
    print(f'Columnas excluidas por no numéricas: '
          f'{[c for c in columnas_potenciales_features if c not in X.columns]}')
    print(X.head())

    # Balance de clases (etiqueta como string, igual que en el resto del pipeline)
    y = df_trabajo['etiqueta_deteccion'].copy()
    print(f'\nBalance de clases:')
    print(y.value_counts())
    if y.isnull().sum() > 0:
        print(f'Advertencia: {y.isnull().sum()} NaNs en y.')
else:
    print('Error: df_trabajo no definido.')

print(f"Features seleccionadas: {X.shape[1]} columnas")
print(f"Columnas: {list(X.columns)}")


## 2. Guardar resultado

In [ ]:
# Guardamos el dataframe completo (con etiquetas + features)
df_trabajo.to_parquet(PARQUET_03, index=False)
print(f"Guardado: {PARQUET_03}")
print(f"Shape: {df_trabajo.shape}")
